In [1]:
%run load_datasets_revar.ipynb

Loaded AMAZON-PHOTOS
Data(x=[7650, 150], edge_index=[2, 238162], y=[7650])
Number of nodes: 7650, Number of features: 150, Number of classes: 8
------------------------------------------------------------
Loaded WIKICS
Data(x=[11701, 150], edge_index=[2, 431726], y=[11701], train_mask=[11701, 20], val_mask=[11701, 20], test_mask=[11701], stopping_mask=[11701, 20])
Number of nodes: 11701, Number of features: 150, Number of classes: 10
------------------------------------------------------------
Loaded PUBMED
Data(x=[19717, 150], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])
Number of nodes: 19717, Number of features: 150, Number of classes: 3
------------------------------------------------------------
Loaded CITESEER
Data(x=[3327, 150], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])
Number of nodes: 3327, Number of features: 150, Number of classes: 6
-----------------------------------------------------

/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/torch_geometric/datasets/wikics.py:45: UserWarning: The WikiCS dataset now returns an undirected graph by default. Please explicitly specify 'is_undirected=False' to restore the old behavior.
  warnings.warn(


In [2]:
import os
import glob
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

from src.utils import compute_accuracy
from src.transform import get_graph_drop_transform
from src.imbalance import Imbalance_
from layers import GNN
from layers.Classifier import Classifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# =============================
# ReVar training & evaluation
# =============================
def train_and_evaluate_revar(data, mask_file, emb_dir, split_name,
                             dataset_name, net="GCN", hidden_dim=150, n_head=1, epochs=200):
    # Show file being processed
    print(f"\n>>> Processing file: {os.path.basename(mask_file)} with encoder: {net}")

    imb = Imbalance_(name=dataset_name, data=data, ratio=0)
    test_mask = imb.split_semi_dataset(mask_file=mask_file)
    train_mask = ~test_mask
    data.train_mask, data.test_mask = train_mask, test_mask

    # Print mask sizes
    print(f"Train size: {train_mask.sum().item()} | Test size: {test_mask.sum().item()}")

    # Encoder
    encoder_model = GNN(
        layer_sizes=[data.num_features, hidden_dim, hidden_dim],
        net=net, n_head=n_head
    ).to(device)
    optimizer = torch.optim.Adam(encoder_model.parameters(), lr=0.01)

    # Encoder training
    transform1 = get_graph_drop_transform(0.2, 0.3)
    transform2 = get_graph_drop_transform(0.2, 0.3)
    data_aug1, data_aug2 = transform1(data), transform2(data)

    start_time = time.time()
    for epoch in range(epochs):
        encoder_model.train()
        optimizer.zero_grad()
        z1, z2 = encoder_model(data_aug1), encoder_model(data_aug2)
        sim_loss = -torch.cosine_similarity(z1, z2, dim=-1).mean()
        sim_loss.backward()
        optimizer.step()
        if (epoch + 1) % 20 == 0:
            print(f"[{net} Encoder] Epoch {epoch+1:03d} | Contrastive Loss: {sim_loss.item():.4f}")
    enc_time = time.time() - start_time

    # Save embeddings
    encoder_model.eval()
    with torch.no_grad():
        embeddings = encoder_model(data).cpu().numpy()
    seed_id = os.path.basename(mask_file).split("seed")[-1].replace(".npy", "")
    emb_file = os.path.join(emb_dir, f"{dataset_name}_{net}_{split_name}_seed{seed_id}.npy")
    np.save(emb_file, embeddings)

    # Classifier
    num_classes = int(data.y.max().item() + 1)
    clf = Classifier(hidden_size=hidden_dim, num_class=num_classes).to(device)
    opt_clf = torch.optim.Adam(clf.parameters(), lr=0.01)
    criterion = torch.nn.CrossEntropyLoss()

    clf_time_start = time.time()
    for epoch in range(200):
        clf.train()
        opt_clf.zero_grad()
        logits, preds = clf(torch.tensor(embeddings).to(device))
        loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        loss.backward()
        opt_clf.step()
        if (epoch + 1) % 20 == 0:
            clf.eval()
            with torch.no_grad():
                _, preds_eval = clf(torch.tensor(embeddings).to(device))
                acc = accuracy_score(data.y[data.test_mask].cpu(),
                                     preds_eval[data.test_mask].cpu())
                f1 = f1_score(data.y[data.test_mask].cpu(),
                              preds_eval[data.test_mask].cpu(),
                              average="macro")
            print(f"[Linear Classifier] Epoch {epoch+1:03d} | Loss: {loss.item():.4f} "
                  f"| Acc: {acc:.4f} | F1: {f1:.4f}")
    clf_time = time.time() - clf_time_start

    # Final Eval
    clf.eval()
    with torch.no_grad():
        logits, preds = clf(torch.tensor(embeddings).to(device))
        _, test_acc, _, test_bacc, _, test_f1 = compute_accuracy(
            preds, data.y, data.train_mask, data.test_mask)

    return {"encoder": net, "seed": seed_id,
            "accuracy": test_acc, "balanced_acc": test_bacc,
            "f1": test_f1, "time": enc_time + clf_time}


# =============================
# Run pipeline
# =============================
def run_pipeline(dataset_name, base_mask_dir, base_emb_dir, base_res_dir):
    data, _ = load_dataset_revar(dataset_name, hidden_dim=150)

    for split in ["70_30", "30_70"]:
        mask_dir = os.path.join(base_mask_dir, dataset_name, split)
        emb_dir = os.path.join(base_emb_dir, dataset_name, split)
        save_dir = os.path.join(base_res_dir, dataset_name, split)
        os.makedirs(emb_dir, exist_ok=True)
        os.makedirs(save_dir, exist_ok=True)

        results = []
        mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.npy")))
        for mask_file in mask_files:
            results.append(train_and_evaluate_revar(data, mask_file, emb_dir, split,
                                                    dataset_name, net="GCN"))
            results.append(train_and_evaluate_revar(data, mask_file, emb_dir, split,
                                                    dataset_name, net="GAT", n_head=8))
            results.append(train_and_evaluate_revar(data, mask_file, emb_dir, split,
                                                    dataset_name, net="SAGE"))

        df = pd.DataFrame(results)
        summary = df.groupby("encoder").agg({
            "accuracy": ["mean", "std"],
            "balanced_acc": ["mean", "std"],
            "f1": ["mean", "std"]
        }).reset_index()
        summary.columns = ["encoder",
                           "accuracy_mean", "accuracy_std",
                           "balanced_acc_mean", "balanced_acc_std",
                           "f1_mean", "f1_std"]

        print(f"\n===== {dataset_name} ({split}) Summary =====")
        for _, r in summary.iterrows():
            print(f"{r['encoder']} -> Acc: {r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f}, "
                  f"BAcc: {r['balanced_acc_mean']:.4f} ± {r['balanced_acc_std']:.4f}, "
                  f"F1: {r['f1_mean']:.4f} ± {r['f1_std']:.4f}")

        # Save summary
        summary.to_csv(os.path.join(save_dir, f"{dataset_name.lower()}_{split}_results.csv"), index=False)

        # Save average execution times
        exec_times = df.groupby("encoder")["time"].mean().reset_index()
        exec_times.rename(columns={"time": "avg_execution_time"}, inplace=True)
        exec_times.to_csv(os.path.join(save_dir, f"{dataset_name.lower()}_{split}_execution_times.csv"), index=False)


# =============================
# Run all datasets
# =============================
base_mask_dir = "/Users/sujan/Modularity based semi supervised learning/masks"
base_emb_dir = "/Users/sujan/Downloads/ReVar/embeddings"
base_res_dir = "/Users/sujan/Downloads/ReVar/results"

datasets = ["Cora", "Citeseer", "Pubmed", "WikiCS"]

for name in datasets:
    run_pipeline(name, base_mask_dir, base_emb_dir, base_res_dir)


Loaded CORA
Data(x=[2708, 150], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Number of nodes: 2708, Number of features: 150, Number of classes: 7

>>> Processing file: Cora_70_30_masked_indices_seed123.npy with encoder: GCN
Train size: 1896 | Test size: 812


/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


[GCN Encoder] Epoch 020 | Contrastive Loss: -0.9342
[GCN Encoder] Epoch 040 | Contrastive Loss: -0.9582
[GCN Encoder] Epoch 060 | Contrastive Loss: -0.9734
[GCN Encoder] Epoch 080 | Contrastive Loss: -0.9843
[GCN Encoder] Epoch 100 | Contrastive Loss: -0.9917
[GCN Encoder] Epoch 120 | Contrastive Loss: -0.9956
[GCN Encoder] Epoch 140 | Contrastive Loss: -0.9974
[GCN Encoder] Epoch 160 | Contrastive Loss: -0.9984
[GCN Encoder] Epoch 180 | Contrastive Loss: -0.9989
[GCN Encoder] Epoch 200 | Contrastive Loss: -0.9992
[Linear Classifier] Epoch 020 | Loss: 1.8100 | Acc: 0.3140 | F1: 0.1043
[Linear Classifier] Epoch 040 | Loss: 1.7817 | Acc: 0.3153 | F1: 0.0889
[Linear Classifier] Epoch 060 | Loss: 1.7672 | Acc: 0.3177 | F1: 0.0969
[Linear Classifier] Epoch 080 | Loss: 1.7565 | Acc: 0.3153 | F1: 0.1152
[Linear Classifier] Epoch 100 | Loss: 1.7471 | Acc: 0.3202 | F1: 0.1257
[Linear Classifier] Epoch 120 | Loss: 1.7383 | Acc: 0.3227 | F1: 0.1321
[Linear Classifier] Epoch 140 | Loss: 1.7301 | A

/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


[GAT Encoder] Epoch 020 | Contrastive Loss: -0.9701
[GAT Encoder] Epoch 040 | Contrastive Loss: -0.9897
[GAT Encoder] Epoch 060 | Contrastive Loss: -0.9959
[GAT Encoder] Epoch 080 | Contrastive Loss: -0.9979
[GAT Encoder] Epoch 100 | Contrastive Loss: -0.9988
[GAT Encoder] Epoch 120 | Contrastive Loss: -0.9990
[GAT Encoder] Epoch 140 | Contrastive Loss: -0.9983
[GAT Encoder] Epoch 160 | Contrastive Loss: -0.9993
[GAT Encoder] Epoch 180 | Contrastive Loss: -0.9996
[GAT Encoder] Epoch 200 | Contrastive Loss: -0.9997
[Linear Classifier] Epoch 020 | Loss: 1.7011 | Acc: 0.3534 | F1: 0.2223
[Linear Classifier] Epoch 040 | Loss: 1.6557 | Acc: 0.3818 | F1: 0.2666
[Linear Classifier] Epoch 060 | Loss: 1.6271 | Acc: 0.3867 | F1: 0.2776
[Linear Classifier] Epoch 080 | Loss: 1.6035 | Acc: 0.3904 | F1: 0.2952
[Linear Classifier] Epoch 100 | Loss: 1.5825 | Acc: 0.3892 | F1: 0.2994
[Linear Classifier] Epoch 120 | Loss: 1.5633 | Acc: 0.4002 | F1: 0.3137
[Linear Classifier] Epoch 140 | Loss: 1.5458 | A

...

[GCN Encoder] Epoch 020 | Contrastive Loss: -0.9575
[GCN Encoder] Epoch 040 | Contrastive Loss: -0.9741
[GCN Encoder] Epoch 060 | Contrastive Loss: -0.9850
[GCN Encoder] Epoch 080 | Contrastive Loss: -0.9918
[GCN Encoder] Epoch 100 | Contrastive Loss: -0.9956
[GCN Encoder] Epoch 120 | Contrastive Loss: -0.9976
[GCN Encoder] Epoch 140 | Contrastive Loss: -0.9987
[GCN Encoder] Epoch 160 | Contrastive Loss: -0.9993
[GCN Encoder] Epoch 180 | Contrastive Loss: -0.9996
[GCN Encoder] Epoch 200 | Contrastive Loss: -0.9997
[Linear Classifier] Epoch 020 | Loss: 2.0522 | Acc: 0.2905 | F1: 0.0823
[Linear Classifier] Epoch 040 | Loss: 2.0176 | Acc: 0.2926 | F1: 0.0829
[Linear Classifier] Epoch 060 | Loss: 2.0025 | Acc: 0.2907 | F1: 0.0824
[Linear Classifier] Epoch 080 | Loss: 1.9948 | Acc: 0.2907 | F1: 0.0824
[Linear Classifier] Epoch 100 | Loss: 1.9900 | Acc: 0.2906 | F1: 0.0823
[Linear Classifier] Epoch 120 | Loss: 1.9867 | Acc: 0.2907 | F1: 0.0828
[Linear Classifier] Epoch 140 | Loss: 1.9839 | A

/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


[GAT Encoder] Epoch 020 | Contrastive Loss: -0.9837
[GAT Encoder] Epoch 040 | Contrastive Loss: -0.9932
[GAT Encoder] Epoch 060 | Contrastive Loss: -0.9971
[GAT Encoder] Epoch 080 | Contrastive Loss: -0.9985
[GAT Encoder] Epoch 100 | Contrastive Loss: -0.9991
[GAT Encoder] Epoch 120 | Contrastive Loss: -0.9994
[GAT Encoder] Epoch 140 | Contrastive Loss: -0.9995
[GAT Encoder] Epoch 160 | Contrastive Loss: -0.9995
[GAT Encoder] Epoch 180 | Contrastive Loss: -0.9996
[GAT Encoder] Epoch 200 | Contrastive Loss: -0.9996
[Linear Classifier] Epoch 020 | Loss: 2.0102 | Acc: 0.2668 | F1: 0.0928
[Linear Classifier] Epoch 040 | Loss: 1.9766 | Acc: 0.3111 | F1: 0.1122
[Linear Classifier] Epoch 060 | Loss: 1.9519 | Acc: 0.3188 | F1: 0.1154
[Linear Classifier] Epoch 080 | Loss: 1.9308 | Acc: 0.3327 | F1: 0.1192
[Linear Classifier] Epoch 100 | Loss: 1.9123 | Acc: 0.3407 | F1: 0.1211
[Linear Classifier] Epoch 120 | Loss: 1.8960 | Acc: 0.3479 | F1: 0.1230
[Linear Classifier] Epoch 140 | Loss: 1.8819 | A

/Users/sujan/miniforge3/envs/tf-gpu/lib/python3.10/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


[SAGE Encoder] Epoch 020 | Contrastive Loss: -0.9973
[SAGE Encoder] Epoch 040 | Contrastive Loss: -0.9993
[SAGE Encoder] Epoch 060 | Contrastive Loss: -0.9997
[SAGE Encoder] Epoch 080 | Contrastive Loss: -0.9998
[SAGE Encoder] Epoch 100 | Contrastive Loss: -0.9999
[SAGE Encoder] Epoch 120 | Contrastive Loss: -0.9999
[SAGE Encoder] Epoch 140 | Contrastive Loss: -0.9999
[SAGE Encoder] Epoch 160 | Contrastive Loss: -1.0000
[SAGE Encoder] Epoch 180 | Contrastive Loss: -1.0000
[SAGE Encoder] Epoch 200 | Contrastive Loss: -1.0000
[Linear Classifier] Epoch 020 | Loss: 2.0750 | Acc: 0.2048 | F1: 0.0756
[Linear Classifier] Epoch 040 | Loss: 2.0314 | Acc: 0.2037 | F1: 0.0710
[Linear Classifier] Epoch 060 | Loss: 2.0061 | Acc: 0.2054 | F1: 0.0723
[Linear Classifier] Epoch 080 | Loss: 1.9879 | Acc: 0.2051 | F1: 0.0755
[Linear Classifier] Epoch 100 | Loss: 1.9733 | Acc: 0.2053 | F1: 0.0782
[Linear Classifier] Epoch 120 | Loss: 1.9612 | Acc: 0.2020 | F1: 0.0784
[Linear Classifier] Epoch 140 | Loss: 